In [1]:
import numpy as np
from plyfile import PlyData
import json
import time
import os

In [6]:
def zero_mean_and_save_npy(ply_filepath, output_dir="processed_data"):
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    base_name = os.path.splitext(os.path.basename(ply_filepath))[0]
    npy_filepath = os.path.join(output_dir, f"{base_name}_normalized.npy")
    meta_filepath = os.path.join(output_dir, f"{base_name}_metadata.json")

    start_time = time.time()
    print(f"Loading PLY file: {ply_filepath}...")
    plydata = PlyData.read(ply_filepath)
    vertex_data = plydata['vertex']
    feature_names = [p.name for p in plydata['vertex'].properties]
    num_points = vertex_data.count
    
    print(f"Loaded {num_points:,} points with features: {feature_names}")

    float32_array = np.zeros((num_points, len(feature_names)), dtype=np.float32)

    # Zero-Mean
    print("Calculating Centroid and performing Zero-Mean Normalization...")
    x_raw = vertex_data['x'][:]
    y_raw = vertex_data['y'][:]
    z_raw = vertex_data['z'][:]

    centroid = {
        "mean_x": float(np.mean(x_raw, dtype=np.float64)),
        "mean_y": float(np.mean(y_raw, dtype=np.float64)),
        "mean_z": float(np.mean(z_raw, dtype=np.float64))
    }

    float32_array[:, feature_names.index('x')] = (x_raw - centroid['mean_x']).astype(np.float32)
    float32_array[:, feature_names.index('y')] = (y_raw - centroid['mean_y']).astype(np.float32)
    float32_array[:, feature_names.index('z')] = (z_raw - centroid['mean_z']).astype(np.float32)

    print("Packing remaining features (RGB, Scalars)...")
    for i, name in enumerate(feature_names):
        if name not in ['x', 'y', 'z']:
            float32_array[:, i] = vertex_data[name][:].astype(np.float32)

    print(f"Saving normalized data to: {npy_filepath}")
    np.save(npy_filepath, float32_array)
    
    print(f"Saving metadata (Centroid) to: {meta_filepath}")
    with open(meta_filepath, 'w') as f:
        json.dump(centroid, f, indent=4)

    process_time = time.time() - start_time
    npy_size = os.path.getsize(npy_filepath) / (1024 * 1024)
    print(f"--- Done in {process_time:.2f} seconds! ---")
    print(f"Output NPY Size: {npy_size:.2f} MB")
    print(f"Centroid Offset: X={centroid['mean_x']:.3f}, Y={centroid['mean_y']:.3f}, Z={centroid['mean_z']:.3f}")

In [7]:
PLY_PATH = os.path.join(os.getcwd(), "..", "data", "raw", "watyaichaimonkol05.ply")

In [8]:
zero_mean_and_save_npy(PLY_PATH)

Loading PLY file: c:\Users\babxk\Documents\GitHub\archaeological-site\notebook\..\data\raw\watyaichaimonkol05.ply...
Loaded 5,490,995 points with features: ['x', 'y', 'z', 'red', 'green', 'blue', 'scalar_Intensity', 'scalar_Original_cloud_index', 'scalar_Roughness_(0.115944)', 'scalar_Mean_curvature_(0.115944)', 'scalar_Normal_change_rate_(0.115944)', 'scalar_Anisotropy_(0.115944)', 'scalar_Planarity_(0.115944)', 'scalar_Linearity_(0.115944)', 'scalar_Surface_variation_(0.115944)']
Calculating Centroid and performing Zero-Mean Normalization...
Packing remaining features (RGB, Scalars)...
Saving normalized data to: processed_data\watyaichaimonkol05_normalized.npy
Saving metadata (Centroid) to: processed_data\watyaichaimonkol05_metadata.json
--- Done in 1.15 seconds! ---
Output NPY Size: 314.20 MB
Centroid Offset: X=9.664, Y=7.257, Z=10.773
